## Ingestion

In [1]:
file = "../data/raw/legal_doc.pdf"

In [18]:
import fitz  # PyMuPDF
import re
import json
import arabic_reshaper
from bidi.algorithm import get_display

# ==========================================
# 1. Text Normalization & Bidi Handling
# ==========================================

def strip_bidi_unicode(text):
    """
    Removes hidden Unicode BiDi control characters (LRE, RLE, PDF, etc.)
    that corrupt word ordering in mixed English/Arabic text.
    """
    if not text:
        return ""
    # Strip Unicode control codes: \u200e, \u200f, \u202a-\u202e, \u2066-\u2069
    return re.sub(r'[\u200e\u200f\u202a-\u202e\u2066-\u2069]', '', text)

def normalize_arabic_numerals(text):
    """Converts Arabic-Indic numerals (١٢٣) to Western digits (123)."""
    if not text:
        return ""
    arabic_to_western = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
    return text.translate(arabic_to_western)

def clean_arabic_for_embedding(text):
    """Removes Tashkeel and normalizes Alefs/Hamzas for embeddings."""
    if not text:
        return ""
    text = re.sub(r'[\u064B-\u065F\u0670]', '', text)  # Remove diacritics
    text = re.sub(r'[أإآ]', 'ا', text)                # Normalize Alef
    text = re.sub(r'[ؤئ]', 'ء', text)                 # Normalize Hamza
    return text.strip()

def fix_arabic_display(text):
    """Reshapes and reverses Arabic text safely for visual display."""
    if not text:
        return ""
    cleaned = strip_bidi_unicode(text)
    reshaped = arabic_reshaper.reshape(cleaned)
    return get_display(reshaped)

def is_repealed(text_ar):
    """Detects if an article has been repealed."""
    repeal_keywords = [r'ألغي[ت]?', r'ملغا[ة]?', r'حذف[ت]?']
    pattern = '|'.join(repeal_keywords)
    return bool(re.search(pattern, text_ar))

# ==========================================
# 2. Parser Logic
# ==========================================

def parse_pdf_articles(file_path):
    doc = fitz.open(file_path)
    print(f"Document opened. Pages: {len(doc)}")

    # Step 1: Collect clean lines page-by-page from respective columns
    ar_lines = []
    en_lines = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        rect = page.rect
        mid_x = rect.width / 2

        # Right column = Arabic, Left column = English
        left_box = fitz.Rect(rect.x0, rect.y0, mid_x, rect.y1)
        right_box = fitz.Rect(mid_x, rect.y0, rect.x1, rect.y1)

        en_page_text = strip_bidi_unicode(page.get_text("text", clip=left_box))
        ar_page_text = strip_bidi_unicode(page.get_text("text", clip=right_box))

        en_lines.extend(en_page_text.splitlines())
        ar_lines.extend(ar_page_text.splitlines())

    # Step 2: Sequential Tracking for Hierarchy & Arabic Articles
    ar_art_pattern = re.compile(r'^\s*(?:ال)?مادة\s*[\(\)\[\]\s]*(\d+)\s*[\(\)\[\]\s]*', re.UNICODE)
    
    # Hierarchy patterns matching Arabic and English headers (e.g. SECTION I)
    book_pattern = re.compile(r'^\s*(?:الكتاب|كتاب|BOOK)\b', re.IGNORECASE)
    chapter_pattern = re.compile(r'^\s*(?:الباب|باب|CHAPTER)\b', re.IGNORECASE)
    section_pattern = re.compile(r'^\s*(?:الفصل|فصل|SECTION|PART)\b', re.IGNORECASE)

    current_book = None
    current_chapter = None
    current_section = None

    ar_articles_map = {}
    ar_hierarchy_map = {}
    
    current_art_num = None
    current_art_text = []

    for raw_line in ar_lines:
        line = normalize_arabic_numerals(raw_line.strip())
        if not line:
            continue

        # Check and update running hierarchy state
        if book_pattern.search(line):
            current_book = line
            continue
        elif chapter_pattern.search(line):
            current_chapter = line
            continue
        elif section_pattern.search(line):
            current_section = line
            continue

        # Check for Article boundary
        art_match = ar_art_pattern.search(line)
        if art_match:
            if current_art_num is not None:
                ar_articles_map[current_art_num] = "\n".join(current_art_text).strip()

            current_art_num = int(art_match.group(1))
            current_art_text = [line]
            
            # Record current active hierarchy state for this article
            ar_hierarchy_map[current_art_num] = {
                "book": current_book,
                "chapter": current_chapter,
                "section": current_section
            }
        elif current_art_num is not None:
            current_art_text.append(line)

    if current_art_num is not None:
        ar_articles_map[current_art_num] = "\n".join(current_art_text).strip()

    # Step 3: Sequential Tracking for English Articles
    en_art_pattern = re.compile(r'Article\s*(\d+)', re.IGNORECASE)
    en_articles_map = {}
    
    current_en_num = None
    current_en_text = []

    for raw_line in en_lines:
        line = normalize_arabic_numerals(raw_line.strip())
        if not line:
            continue

        art_match = en_art_pattern.search(line)
        if art_match:
            if current_en_num is not None:
                en_articles_map[current_en_num] = " ".join(current_en_text).strip()
            current_en_num = int(art_match.group(1))
            current_en_text = [line]
        elif current_en_num is not None:
            current_en_text.append(line)

    if current_en_num is not None:
        en_articles_map[current_en_num] = " ".join(current_en_text).strip()

    # Step 4: Merge Arabic & English by Article Number
    all_art_nums = sorted(list(set(ar_articles_map.keys()).union(set(en_articles_map.keys()))))
    articles = []

    for art_num in all_art_nums:
        ar_raw = ar_articles_map.get(art_num, "")
        en_raw = en_articles_map.get(art_num, "")
        hierarchy = ar_hierarchy_map.get(art_num, {"book": None, "chapter": None, "section": None})

        articles.append({
            "article_number": art_num,
            "book": hierarchy["book"],
            "chapter": hierarchy["chapter"],
            "section": hierarchy["section"],
            "text_ar": clean_arabic_for_embedding(ar_raw),
            "text_ar_display": fix_arabic_display(ar_raw),
            "text_en": re.sub(r'\s+', ' ', en_raw).strip(),
            "is_repealed": is_repealed(ar_raw),
            "citation": f"Egyptian Civil Code, Article {art_num}"
        })

    return articles

# ==========================================
# 3. Execution
# ==========================================

parsed_articles = parse_pdf_articles(file)

print(f"\nSuccessfully extracted {len(parsed_articles)} articles.")

if parsed_articles:
    print("\n--- Sample Article Output ---")
    sample = parsed_articles[0].copy()
    display_text = sample.pop("text_ar_display")
    print(json.dumps(sample, ensure_ascii=False, indent=4))
    print("\nCorrect Visual Arabic Display:")
    print(display_text[:200] + "...")

Document opened. Pages: 170

Successfully extracted 1095 articles.

--- Sample Article Output ---
{
    "article_number": 1,
    "book": null,
    "chapter": null,
    "section": null,
    "text_ar": "",
    "text_en": "Article 1 Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.",
    "is_repealed": false,
    "citation": "Egyptian Civil Code, Article 1"
}

Correct Visual Arabic Display:
...


In [20]:
parsed_articles

[{'article_number': 1,
  'book': None,
  'chapter': None,
  'section': None,
  'text_ar': '',
  'text_ar_display': '',
  'text_en': 'Article 1 Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.',
  'is_repealed': False,
  'citation': 'Egyptian Civil Code, Article 1'},
 {'article_number': 2,
  'book': None,
  'chapter': None,
  'section': None,
  'text_ar': '',
  'text_ar_display': '',
  'text_en': 'Article 2 A provision of a law can only be repealed by a subsequent law expressly providing for such repeal, or containing a provision inconsistent with a provision of the former law or regulating anew a matter previously regulated by a former law.',
  'is_repeale

In [53]:
import json
import re
import pdfplumber

# ==========================================
# 1. ARABIC REVERSAL & NORMALIZATION
# ==========================================

def fix_arabic_token(token: str) -> str:
    """
    Reverses character order and swaps brackets for Arabic tokens
    extracted from legacy LTR-encoded PDFs.
    """
    # Swap parentheses and brackets after reversing character sequence
    bracket_map = str.maketrans("()[]{}<>", ")(][}{><")
    return token[::-1].translate(bracket_map)


def process_arabic_row_words(right_words) -> str:
    """
    1. Sorts Arabic words from Right-to-Left (descending x0 coordinate).
    2. Reverses character sequence of each word to restore proper spelling.
    3. Cleans up Tashkeel, Tatweel, and normalizes digits/Alef forms.
    """
    if not right_words:
        return ""

    # Sort Right-to-Left (highest x0 coordinate first)
    sorted_right = sorted(right_words, key=lambda w: w['x0'], reverse=True)

    fixed_tokens = [fix_arabic_token(w['text']) for w in sorted_right]
    text = " ".join(fixed_tokens)

    # 1. Convert Eastern Arabic numerals (٠١٢٣) to standard digits (0123)
    text = text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))

    # 2. Strip Unicode directional markers
    text = re.sub(r'[\u200e\u200f\u202a-\u202e\u200c\u200d]', ' ', text)

    # 3. Remove Tashkeel (diacritics) & Tatweel (ـ)
    text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)
    text = re.sub(r'ـ', '', text)

    # 4. Normalize Alef forms (أ, إ, آ -> ا) and Alef Maksura (ى -> ي)
    text = re.sub(r'[إأآٱ]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)

    # Clean multi-spaces
    return re.sub(r'\s+', ' ', text).strip()


def clean_english_text(text: str) -> str:
    if not text:
        return ""
    return re.sub(r'\s+', ' ', text).strip()


# ==========================================
# 2. ROW GROUPING & PARSER
# ==========================================

def group_words_into_rows(words, y_tolerance=4):
    """Groups words into horizontal lines based on y-coordinates."""
    words_sorted = sorted(words, key=lambda w: (w['top'], w['x0']))
    rows = []
    current_row = []
    current_top = None

    for w in words_sorted:
        if current_top is None or abs(w['top'] - current_top) <= y_tolerance:
            current_row.append(w)
            if current_top is None:
                current_top = w['top']
        else:
            rows.append(current_row)
            current_row = [w]
            current_top = w['top']

    if current_row:
        rows.append(current_row)
    return rows


def parse_two_column_pdf(pdf_path: str):
    articles = []

    current_book = ""
    current_chapter = ""
    current_section = ""
    current_topic = ""
    current_article = None

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            words = page.extract_words(x_tolerance=3, y_tolerance=3)
            if not words:
                continue

            # Skip the preamble block on Page 1 before SECTION I / الفصل الأول
            if page_num == 1:
                start_y = None
                for w in words:
                    fixed_w = fix_arabic_token(w['text'])
                    if "SECTION" in w['text'].upper() or "الفصل" in fixed_w:
                        start_y = w['top'] - 5
                        break
                if start_y is not None:
                    words = [w for w in words if w['top'] >= start_y]
                else:
                    words = [w for w in words if w['top'] >= page.height * 0.20]

            mid_x = page.width / 2.0
            rows = group_words_into_rows(words)

            for row in rows:
                # Divide words into Left (English) and Right (Arabic) columns
                left_words = sorted([w for w in row if w['x1'] <= mid_x + 10], key=lambda w: w['x0'])
                right_words = [w for w in row if w['x0'] >= mid_x - 10]

                line_en = clean_english_text(" ".join(w['text'] for w in left_words))
                line_ar = process_arabic_row_words(right_words)

                if not line_en and not line_ar:
                    continue

                # Metadata updates
                upper_en = line_en.upper()
                if "BOOK" in upper_en and any(b in upper_en for b in ["BOOK I", "BOOK II", "BOOK III", "BOOK IV"]):
                    current_book = line_en
                    continue
                elif "CHAPTER" in upper_en:
                    current_chapter = line_en
                    continue
                elif "SECTION" in upper_en:
                    current_section = line_en
                    continue
                elif re.match(r'^\d+\.\s+[A-Z]', line_en):
                    current_topic = line_en
                    continue

                # Article header detection
                art_en_match = re.search(r'\bArticle\s+(\d+)\b', line_en, re.IGNORECASE)
                art_ar_match = re.search(r'(?:مادة|المادة)\s*\(?\s*(\d+)\s*\)?', line_ar)

                art_num = None
                if art_en_match:
                    art_num = int(art_en_match.group(1))
                elif art_ar_match:
                    art_num = int(art_ar_match.group(1))

                if art_num is not None:
                    # Save previous article
                    if current_article:
                        current_article["text_en"] = clean_english_text(current_article["text_en"])
                        current_article["text_ar"] = clean_english_text(current_article["text_ar"])
                        current_article["is_repealed"] = ("repealed" in current_article["text_en"].lower() or 
                                                          "الغيت" in current_article["text_ar"])
                        articles.append(current_article)

                    # Start new article
                    current_article = {
                        "article_number": art_num,
                        "book": current_book,
                        "chapter": current_chapter,
                        "section": current_section,
                        "topic": current_topic,
                        "text_ar": line_ar,
                        "text_en": line_en,
                        "is_repealed": False,
                        "source_page": page_num,
                        "citation": f"Egyptian Civil Code, Article {art_num}"
                    }
                else:
                    # Append row text synchronously
                    if current_article:
                        if line_en:
                            current_article["text_en"] += " " + line_en
                        if line_ar:
                            current_article["text_ar"] += " " + line_ar

        # Save final article
        if current_article:
            current_article["text_en"] = clean_english_text(current_article["text_en"])
            current_article["text_ar"] = clean_english_text(current_article["text_ar"])
            current_article["is_repealed"] = ("repealed" in current_article["text_en"].lower() or 
                                              "الغيت" in current_article["text_ar"])
            articles.append(current_article)

    return articles


# ==========================================
# 3. EXECUTION & SAVE
# ==========================================

if __name__ == "__main__":
    results = parse_two_column_pdf(file)

    with open("extracted_articles.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    if results:
        print(json.dumps(results[0], ensure_ascii=False, indent=2))

{
  "article_number": 1,
  "book": "",
  "chapter": "",
  "section": "SECTION I",
  "topic": "1. Laws and Rights",
  "text_ar": "مادة ) 1 ) (1) تسري النصوص التشريعية علي جميع المسائل التي تتناولها هذه النصوص في لفظها او في فحواها. )2( فاذا لم يوجد نص تشريعي يمكن تطبيقه، حكم القاضي بمقتضي العرف، فاذا لم يوجد، فبمقتضي مبادئ الشريعة االسالمية، فاذا لم يوجد فبمقتضي مبادئ القانون الطبيعي وقواعد العدالة.",
  "text_en": "Article 1 Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.",
  "is_repealed": false,
  "source_page": 1,
  "citation": "Egyptian Civil Code, Article 1"
}


In [24]:
import json

def analyze_extracted_json(json_file_path):
    # 1. تحميل ملف الـ JSON
    with open(json_file_path, 'r', encoding='utf-8') as f:
        articles = json.load(f)

    # 2. استخراج المواد الملغاة (is_repealed: True)
    repealed_articles = [art for art in articles if art.get("is_repealed") is True]

    # 3. حصر كافة أرقام المواد الموجودة في الملف
    extracted_numbers = set(art["article_number"] for art in articles if "article_number" in art)

    # 4. تحديد الأرقام المفقودة بين 1 و 1149
    all_expected_numbers = set(range(1, 1150)) # من 1 إلى 1149
    missing_numbers = sorted(list(all_expected_numbers - extracted_numbers))

    # 5. تجميع الأرقام المفقودة في فترات متصلة (Ranges)
    missing_ranges = []
    if missing_numbers:
        start = missing_numbers[0]
        end = missing_numbers[0]

        for num in missing_numbers[1:]:
            if num == end + 1:
                end = num
            else:
                if start == end:
                    missing_ranges.append(f"المادة {start}")
                else:
                    missing_ranges.append(f"المواد من {start} إلى {end}")
                start = num
                end = num
        
        # إضافة الفترة الأخيرة
        if start == end:
            missing_ranges.append(f"المادة {start}")
        else:
            missing_ranges.append(f"المواد من {start} إلى {end}")

    # 6. طباعة النتائج بشكل منظم
    print("=" * 50)
    print("1. المواد الملغاة (is_repealed: True):")
    print("=" * 50)
    print(f"إجمالي عدد المواد الملغاة: {len(repealed_articles)}")
    for art in repealed_articles:
        print(f" - مادة {art['article_number']} (الصفحة {art.get('source_page', 'N/A')})")

    print("\n" + "=" * 50)
    print("2. المواد المفقودة/غير الموجودة بين [1 - 1149]:")
    print("=" * 50)
    print(f"إجمالي الأرقام المفقودة: {len(missing_numbers)} مادة")
    print("فترات المواد المفقودة:")
    for r in missing_ranges:
        print(f" - {r}")

    # إرجاع النتائج لاستخدامها برمجياً إن أردت
    return {
        "repealed_articles_count": len(repealed_articles),
        "repealed_numbers": [a["article_number"] for a in repealed_articles],
        "missing_numbers_count": len(missing_numbers),
        "missing_numbers_list": missing_numbers,
        "missing_ranges": missing_ranges
    }

if __name__ == "__main__":
    results = analyze_extracted_json("output_4.json")

1. المواد الملغاة (is_repealed: True):
إجمالي عدد المواد الملغاة: 0

2. المواد المفقودة/غير الموجودة بين [1 - 1149]:
إجمالي الأرقام المفقودة: 56 مادة
فترات المواد المفقودة:
 - المواد من 55 إلى 80
 - المواد من 389 إلى 417
 - المادة 452


In [12]:
import re
import json
import pdfplumber

def convert_to_eastern_arabic_digits(num_str: str) -> str:
    """Converts Western digits (0-9) to Eastern Arabic digits (٠-٩)."""
    western_to_eastern = str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")
    return num_str.translate(western_to_eastern)

def extract_english_only(text: str) -> str:
    """Removes Arabic characters and extra spaces, returning only English metadata."""
    cleaned = re.sub(r'[\u0600-\u06FF]+', '', text)
    return " ".join(cleaned.split()).strip()

def fix_reversed_arabic_text(text: str) -> str:
    """
    Fixes character-level reversal in Arabic tokens (e.g., 'ةدام' -> 'مادة'),
    preserves LTR order for numbers (preventing '204' from becoming '402'),
    and converts digits to Eastern Arabic numerals (٠-٩).
    """
    if not text:
        return ""

    tokens = text.split()
    fixed_tokens = []

    for token in tokens:
        # Check if token consists purely of digits
        if re.match(r'^[0-9\u0660-\u0669\.,\-]+$', token):
            fixed_tokens.append(convert_to_eastern_arabic_digits(token))
            continue

        # If token contains Arabic characters, reverse the character sequence
        if re.search(r'[\u0600-\u06FF]', token):
            reversed_token = token[::-1]

            # Re-reverse any embedded digits so multi-digit numbers read LTR (e.g., 204 stays 204)
            def restore_digit_direction(match):
                return match.group(0)[::-1]

            corrected_token = re.sub(r'[\d\u0660-\u0669]+', restore_digit_direction, reversed_token)
            fixed_tokens.append(convert_to_eastern_arabic_digits(corrected_token))
        else:
            fixed_tokens.append(token)

    return " ".join(fixed_tokens)


def extract_legal_document(pdf_path: str, output_json_path: str):
    print(f"--- Starting Extraction: {pdf_path} ---")

    current_book = ""
    current_chapter = ""
    current_section = ""
    current_section_name = ""
    current_topic = ""
    expecting_section_name = False

    extracted_articles = []
    current_article = None

    # Flexible Regex Patterns
    re_book = re.compile(r'^(BOOK|الكتاب)\b', re.IGNORECASE)
    re_chapter = re.compile(r'^(CHAPTER|الباب)\b', re.IGNORECASE)
    re_section = re.compile(r'^(SECTION|الفرع|القسم)\b', re.IGNORECASE)

    # Matches: "Article 1", "ARTICLE (1)", "مادة 1", "المادة 1"
    re_article_en = re.compile(r'\b(?:Article|ARTICLE)\s*\(?\s*(\d+)\s*\)?')
    re_article_ar = re.compile(r'\b(?:مادة|المادة|مادّة)\s*\(?\s*([\d\u0660-\u0669]+)\s*\)?')

    with pdfplumber.open(pdf_path) as pdf:
        print(f"Total Pages: {len(pdf.pages)}")

        for page_num, page in enumerate(pdf.pages, start=1):
            words = page.extract_words(extra_attrs=["fontname"])
            if not words:
                continue

            # Sort words vertically top-to-bottom
            words_sorted = sorted(words, key=lambda w: w["top"])

            # Group words into physical visual rows (3.5pt tolerance)
            lines = []
            for w in words_sorted:
                placed = False
                for line in lines:
                    if abs(w["top"] - line[0]["top"]) < 3.5:
                        line.append(w)
                        placed = True
                        break
                if not placed:
                    lines.append([w])

            for line_words in lines:
                # Sort words within row horizontally (Left to Right)
                line_words.sort(key=lambda w: w["x0"])
                line_text_raw = " ".join(w["text"] for w in line_words).strip()

                if not line_text_raw:
                    continue

                is_bold = any("bold" in str(w.get("fontname", "")).lower() for w in line_words)

                # Separate English and Arabic text components
                ar_words = [w["text"] for w in line_words if re.search(r'[\u0600-\u06FF]', w["text"])]
                en_words = [w["text"] for w in line_words if not re.search(r'[\u0600-\u06FF]', w["text"])]

                line_en_raw = " ".join(en_words).strip()
                line_ar_raw = " ".join(ar_words).strip()

                # 1. BOOK DETECTION (English Only)
                if re_book.search(line_text_raw):
                    current_book = extract_english_only(line_text_raw)
                    current_chapter = ""
                    current_section = ""
                    current_section_name = ""
                    current_topic = ""
                    expecting_section_name = False
                    continue

                # 2. CHAPTER DETECTION (English Only)
                if re_chapter.search(line_text_raw):
                    current_chapter = extract_english_only(line_text_raw)
                    current_section = ""
                    current_section_name = ""
                    current_topic = ""
                    expecting_section_name = False
                    continue

                # 3. SECTION DETECTION (English Only)
                if re_section.search(line_text_raw):
                    current_section = extract_english_only(line_text_raw)
                    current_section_name = ""
                    current_topic = ""
                    expecting_section_name = True
                    continue

                # 4. CAPTURE SECTION NAME (Row immediately following SECTION)
                if expecting_section_name:
                    if not (re_article_en.search(line_text_raw) or re_article_ar.search(line_text_raw)):
                        current_section_name = extract_english_only(line_text_raw)
                        expecting_section_name = False
                        continue
                    expecting_section_name = False

                # 5. ARTICLE START DETECTION
                match_en = re_article_en.search(line_text_raw)
                match_ar = re_article_ar.search(line_text_raw)

                if match_en or match_ar:
                    if current_article:
                        extracted_articles.append(current_article)

                    art_num_str = match_en.group(1) if match_en else match_ar.group(1)
                    art_num = int(art_num_str.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")))

                    current_article = {
                        "article_number": art_num,
                        "book": current_book,
                        "chapter": current_chapter,
                        "section": current_section,
                        "section_name": current_section_name,
                        "topic": current_topic,
                        "text_ar": fix_reversed_arabic_text(line_ar_raw),
                        "text_en": line_en_raw,
                        "is_repealed": "ملغاة" in line_text_raw or "repealed" in line_text_raw.lower(),
                        "source_page": page_num
                    }
                    continue

                # 6. TOPIC DETECTION (English Only)
                if is_bold and current_article is None:
                    current_topic = extract_english_only(line_text_raw)
                    continue

                # 7. CONTINUATION TEXT APPENDING
                if current_article is not None:
                    if line_ar_raw:
                        fixed_ar = fix_reversed_arabic_text(line_ar_raw)
                        current_article["text_ar"] = (current_article["text_ar"] + " " + fixed_ar).strip()
                    if line_en_raw:
                        current_article["text_en"] = (current_article["text_en"] + " " + line_en_raw).strip()

        if current_article:
            extracted_articles.append(current_article)

    # Deduplicate and sort articles
    unique_articles = {}
    for art in extracted_articles:
        num = art["article_number"]
        if num not in unique_articles:
            unique_articles[num] = art
        else:
            if art["text_ar"] and not unique_articles[num]["text_ar"]:
                unique_articles[num]["text_ar"] = art["text_ar"]
            if art["text_en"] and not unique_articles[num]["text_en"]:
                unique_articles[num]["text_en"] = art["text_en"]

    final_list = [unique_articles[k] for k in sorted(unique_articles.keys())]

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(final_list, f, ensure_ascii=False, indent=2)

    print(f"--- Completed! Extracted {len(final_list)} articles into '{output_json_path}' ---")
    return final_list

extract_legal_document(file, "output_2.json")

--- Starting Extraction: ../data/raw/legal_doc.pdf ---
Total Pages: 170
--- Completed! Extracted 1093 articles into 'output_2.json' ---


[{'article_number': 1,
  'book': '',
  'chapter': '',
  'section': 'SECTION I',
  'section_name': 'Laws and their Applications',
  'topic': '1. Laws and Rights -',
  'text_ar': '١ مادة هذه تتناولها التي المسائل جميع على التشريعية النصوص تسرى ١ فحواها. في أو لفظها في النصوص بمقتضى القاضي حكم تطبيقه، يمكن تشريعي نص يوجد لم فإذا (٢) يوجد لم فإذا اإلسالمية، الشريعة مبادئ فبمقتضى يوجد، لم فإذا العرف، العدالة وقواعد الطبيعي القانون مبادئ فبمقتضى',
  'text_en': 'Article 1 ( ( Provisions of laws govern all matters to which these ( ) provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. . In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.',
  'is_repealed': False,
  'source_page': 1},
 {'article_number': 2,
  'book': '',
  'chapter': '',
  'section': 'SECTION I',
  'secti

In [21]:
import re
import json
import pdfplumber

def convert_to_eastern_arabic_digits(num_str: str) -> str:
    """Converts Western digits (0-9) to Eastern Arabic digits (٠-٩)."""
    western_to_eastern = str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")
    return num_str.translate(western_to_eastern)

def extract_english_only(text: str) -> str:
    """Removes Arabic characters and extra spaces, returning only English metadata."""
    cleaned = re.sub(r'[\u0600-\u06FF]+', '', text)
    return " ".join(cleaned.split()).strip()

def fix_visual_arabic_word(word: str) -> str:
    """
    Reverses character order for visual Arabic words (e.g., 'ةدام' -> 'مادة'),
    while preserving digit sequences and formatting.
    """
    if not word:
        return ""
    
    # Preserve standalone digits / punctuation direction
    if re.match(r'^[0-9\u0660-\u0669\.,\(\)\-]+$', word):
        return convert_to_eastern_arabic_digits(word)
    
    # Reverse character sequence for visual Arabic words
    if re.search(r'[\u0600-\u06FF]', word):
        reversed_word = word[::-1]
        
        # Restore embedded multi-digit numbers to LTR reading order
        def restore_digits(match):
            return match.group(0)[::-1]
        
        corrected = re.sub(r'[\d\u0660-\u0669]+', restore_digits, reversed_word)
        return convert_to_eastern_arabic_digits(corrected)
    
    return word

def process_arabic_line(ar_words: list) -> str:
    """
    Sorts Arabic words from RIGHT to LEFT (descending x0 coordinate)
    to restore true Arabic sentence word order, then fixes letter sequence.
    """
    if not ar_words:
        return ""
    
    # Sort Right-to-Left (highest x0 coordinate to lowest)
    ar_words_rtl = sorted(ar_words, key=lambda w: w["x0"], reverse=True)
    
    fixed_words = [fix_visual_arabic_word(w["text"]) for w in ar_words_rtl]
    return " ".join(fixed_words).strip()

def process_english_line(en_words: list) -> str:
    """Sorts English words Left-to-Right (ascending x0 coordinate)."""
    if not en_words:
        return ""
    en_words_ltr = sorted(en_words, key=lambda w: w["x0"])
    return " ".join(w["text"] for w in en_words_ltr).strip()


def extract_legal_document(pdf_path: str, output_json_path: str):
    print(f"--- Starting Extraction: {pdf_path} ---")

    current_book = ""
    current_chapter = ""
    current_section = ""
    current_section_name = ""
    current_topic = ""
    expecting_section_name = False

    extracted_articles = []
    current_article = None

    # Flexible Regex Patterns
    re_book = re.compile(r'^(BOOK|الكتاب)\b', re.IGNORECASE)
    re_chapter = re.compile(r'^(CHAPTER|الباب)\b', re.IGNORECASE)
    re_section = re.compile(r'^(SECTION|الفرع|القسم)\b', re.IGNORECASE)

    # Article header matches
    re_article_en = re.compile(r'\b(?:Article|ARTICLE)\s*\(?\s*(\d+)\s*\)?')
    re_article_ar = re.compile(r'\b(?:مادة|المادة|مادّة)\s*\(?\s*([\d\u0660-\u0669]+)\s*\)?')

    with pdfplumber.open(pdf_path) as pdf:
        print(f"Total Pages: {len(pdf.pages)}")

        for page_num, page in enumerate(pdf.pages, start=1):
            words = page.extract_words(extra_attrs=["fontname"])
            if not words:
                continue

            # Sort top-to-bottom
            words_sorted = sorted(words, key=lambda w: w["top"])

            # Group words into visual lines (3.5pt height tolerance)
            lines = []
            for w in words_sorted:
                placed = False
                for line in lines:
                    if abs(w["top"] - line[0]["top"]) < 3.5:
                        line.append(w)
                        placed = True
                        break
                if not placed:
                    lines.append([w])

            for line_words in lines:
                line_words_ltr = sorted(line_words, key=lambda w: w["x0"])
                line_text_raw = " ".join(w["text"] for w in line_words_ltr).strip()

                if not line_text_raw:
                    continue

                is_bold = any("bold" in str(w.get("fontname", "")).lower() for w in line_words)

                # Separate language components
                ar_word_objs = [w for w in line_words if re.search(r'[\u0600-\u06FF]', w["text"])]
                en_word_objs = [w for w in line_words if not re.search(r'[\u0600-\u06FF]', w["text"])]

                line_ar_str = process_arabic_line(ar_word_objs)
                line_en_str = process_english_line(en_word_objs)

                # 1. BOOK DETECTION (English Only)
                if re_book.search(line_text_raw):
                    current_book = extract_english_only(line_text_raw)
                    current_chapter = ""
                    current_section = ""
                    current_section_name = ""
                    current_topic = ""
                    expecting_section_name = False
                    continue

                # 2. CHAPTER DETECTION (English Only)
                if re_chapter.search(line_text_raw):
                    current_chapter = extract_english_only(line_text_raw)
                    current_section = ""
                    current_section_name = ""
                    current_topic = ""
                    expecting_section_name = False
                    continue

                # 3. SECTION DETECTION (English Only)
                if re_section.search(line_text_raw):
                    current_section = extract_english_only(line_text_raw)
                    current_section_name = ""
                    current_topic = ""
                    expecting_section_name = True
                    continue

                # 4. CAPTURE SECTION NAME
                if expecting_section_name:
                    if not (re_article_en.search(line_text_raw) or re_article_ar.search(line_text_raw)):
                        current_section_name = extract_english_only(line_text_raw)
                        expecting_section_name = False
                        continue
                    expecting_section_name = False

                # 5. ARTICLE START DETECTION
                match_en = re_article_en.search(line_text_raw)
                match_ar = re_article_ar.search(line_text_raw)

                if match_en or match_ar:
                    if current_article:
                        extracted_articles.append(current_article)

                    art_num_str = match_en.group(1) if match_en else match_ar.group(1)
                    art_num = int(art_num_str.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")))

                    current_article = {
                        "article_number": art_num,
                        "book": current_book,
                        "chapter": current_chapter,
                        "section": current_section,
                        "section_name": current_section_name,
                        "topic": current_topic,
                        "text_ar": line_ar_str,
                        "text_en": line_en_str,
                        "is_repealed": "ملغاة" in line_text_raw or "repealed" in line_text_raw.lower(),
                        "source_page": page_num
                    }
                    continue

                # 6. TOPIC DETECTION (English Only)
                if is_bold and current_article is None:
                    current_topic = extract_english_only(line_text_raw)
                    continue

                # 7. CONTINUATION TEXT APPENDING
                if current_article is not None:
                    if line_ar_str:
                        current_article["text_ar"] = (current_article["text_ar"] + " " + line_ar_str).strip()
                    if line_en_str:
                        current_article["text_en"] = (current_article["text_en"] + " " + line_en_str).strip()

        if current_article:
            extracted_articles.append(current_article)

    # Deduplicate and sort articles
    unique_articles = {}
    for art in extracted_articles:
        num = art["article_number"]
        if num not in unique_articles:
            unique_articles[num] = art
        else:
            if art["text_ar"] and not unique_articles[num]["text_ar"]:
                unique_articles[num]["text_ar"] = art["text_ar"]
            if art["text_en"] and not unique_articles[num]["text_en"]:
                unique_articles[num]["text_en"] = art["text_en"]

    final_list = [unique_articles[k] for k in sorted(unique_articles.keys())]

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(final_list, f, ensure_ascii=False, indent=2)

    print(f"--- Completed! Extracted {len(final_list)} articles into '{output_json_path}' ---")
    return final_list
extract_legal_document(file, "output_3.json")

--- Starting Extraction: ../data/raw/legal_doc.pdf ---
Total Pages: 170
--- Completed! Extracted 1093 articles into 'output_3.json' ---


[{'article_number': 1,
  'book': '',
  'chapter': '',
  'section': 'SECTION I',
  'section_name': 'Laws and their Applications',
  'topic': '1. Laws and Rights -',
  'text_ar': 'مادة ١ ١ تسرى النصوص التشريعية على جميع المسائل التي تتناولها هذه النصوص في لفظها أو في فحواها. )٢( فإذا لم يوجد نص تشريعي يمكن تطبيقه، حكم القاضي بمقتضى العرف، فإذا لم يوجد، فبمقتضى مبادئ الشريعة اإلسالمية، فإذا لم يوجد فبمقتضى مبادئ القانون الطبيعي وقواعد العدالة',
  'text_en': 'Article 1 ( ( Provisions of laws govern all matters to which these ( ) provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. . In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.',
  'is_repealed': False,
  'source_page': 1},
 {'article_number': 2,
  'book': '',
  'chapter': '',
  'section': 'SECTION I',
  'secti

In [23]:
import re
import json
import pdfplumber

def convert_to_eastern_arabic_digits(num_str: str) -> str:
    """Converts Western digits (0-9) to Eastern Arabic digits (٠-٩)."""
    western_to_eastern = str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")
    return num_str.translate(western_to_eastern)

def extract_english_only(text: str) -> str:
    """Removes Arabic characters and extra spaces, returning only English metadata."""
    cleaned = re.sub(r'[\u0600-\u06FF]+', '', text)
    return " ".join(cleaned.split()).strip()

def fix_visual_arabic_word(word: str) -> str:
    """
    Reverses character order for visual Arabic words (e.g., 'ةدام' -> 'مادة'),
    while preserving digit sequences and formatting.
    """
    if not word:
        return ""
    
    # Preserve standalone digits / punctuation direction
    if re.match(r'^[0-9\u0660-\u0669\.,\(\)\-]+$', word):
        return convert_to_eastern_arabic_digits(word)
    
    # Reverse character sequence for visual Arabic words
    if re.search(r'[\u0600-\u06FF]', word):
        reversed_word = word[::-1]
        
        # Restore embedded multi-digit numbers to LTR reading order
        def restore_digits(match):
            return match.group(0)[::-1]
        
        corrected = re.sub(r'[\d\u0660-\u0669]+', restore_digits, reversed_word)
        return convert_to_eastern_arabic_digits(corrected)
    
    return word

def process_arabic_line(ar_words: list) -> str:
    """
    Sorts Arabic words from RIGHT to LEFT (descending x0 coordinate)
    to restore true Arabic sentence word order, then fixes letter sequence.
    """
    if not ar_words:
        return ""
    
    # Sort Right-to-Left (highest x0 coordinate to lowest)
    ar_words_rtl = sorted(ar_words, key=lambda w: w["x0"], reverse=True)
    
    fixed_words = [fix_visual_arabic_word(w["text"]) for w in ar_words_rtl]
    return " ".join(fixed_words).strip()

def process_english_line(en_words: list) -> str:
    """Sorts English words Left-to-Right (ascending x0 coordinate)."""
    if not en_words:
        return ""
    en_words_ltr = sorted(en_words, key=lambda w: w["x0"])
    return " ".join(w["text"] for w in en_words_ltr).strip()


def extract_legal_document(pdf_path: str, output_json_path: str):
    print(f"--- Starting Extraction: {pdf_path} ---")

    current_book = ""
    current_chapter = ""
    current_section = ""
    current_section_name = ""
    current_topic = ""
    expecting_section_name = False

    extracted_articles = []
    current_article = None

    # Flexible Regex Patterns
    re_book = re.compile(r'^(BOOK|الكتاب)\b', re.IGNORECASE)
    re_chapter = re.compile(r'^(CHAPTER|الباب)\b', re.IGNORECASE)
    re_section = re.compile(r'^(SECTION|الفرع|القسم)\b', re.IGNORECASE)

    # Article header matches
    re_article_en = re.compile(r'\b(?:Article|ARTICLE)\s*\(?\s*(\d+)\s*\)?')
    re_article_ar = re.compile(r'\b(?:مادة|المادة|مادّة)\s*\(?\s*([\d\u0660-\u0669]+)\s*\)?')

    # Topic index pattern (e.g., "1. ", "2. ", "1 -", "(1)")
    re_topic_number = re.compile(r'^\s*\d+[\.\s\-]', re.IGNORECASE)

    with pdfplumber.open(pdf_path) as pdf:
        print(f"Total Pages: {len(pdf.pages)}")

        for page_num, page in enumerate(pdf.pages, start=1):
            words = page.extract_words(extra_attrs=["fontname"])
            if not words:
                continue

            # Sort top-to-bottom
            words_sorted = sorted(words, key=lambda w: w["top"])

            # Group words into visual lines (3.5pt height tolerance)
            lines = []
            for w in words_sorted:
                placed = False
                for line in lines:
                    if abs(w["top"] - line[0]["top"]) < 3.5:
                        line.append(w)
                        placed = True
                        break
                if not placed:
                    lines.append([w])

            for line_words in lines:
                line_words_ltr = sorted(line_words, key=lambda w: w["x0"])
                line_text_raw = " ".join(w["text"] for w in line_words_ltr).strip()

                if not line_text_raw:
                    continue

                is_bold = any("bold" in str(w.get("fontname", "")).lower() for w in line_words)

                # Separate language components
                ar_word_objs = [w for w in line_words if re.search(r'[\u0600-\u06FF]', w["text"])]
                en_word_objs = [w for w in line_words if not re.search(r'[\u0600-\u06FF]', w["text"])]

                line_ar_str = process_arabic_line(ar_word_objs)
                line_en_str = process_english_line(en_word_objs)

                # 1. BOOK DETECTION (English Only)
                if re_book.search(line_text_raw):
                    if current_article:
                        extracted_articles.append(current_article)
                        current_article = None
                    current_book = extract_english_only(line_text_raw)
                    current_chapter = ""
                    current_section = ""
                    current_section_name = ""
                    current_topic = ""
                    expecting_section_name = False
                    continue

                # 2. CHAPTER DETECTION (English Only)
                if re_chapter.search(line_text_raw):
                    if current_article:
                        extracted_articles.append(current_article)
                        current_article = None
                    current_chapter = extract_english_only(line_text_raw)
                    current_section = ""
                    current_section_name = ""
                    current_topic = ""
                    expecting_section_name = False
                    continue

                # 3. SECTION DETECTION (English Only)
                if re_section.search(line_text_raw):
                    if current_article:
                        extracted_articles.append(current_article)
                        current_article = None
                    current_section = extract_english_only(line_text_raw)
                    current_section_name = ""
                    current_topic = ""
                    expecting_section_name = True
                    continue

                # 4. CAPTURE SECTION NAME OR TOPIC RIGHT AFTER SECTION
                if expecting_section_name:
                    match_en = re_article_en.search(line_text_raw)
                    match_ar = re_article_ar.search(line_text_raw)
                    if not (match_en or match_ar):
                        if re_topic_number.match(line_en_str) or re_topic_number.match(line_text_raw):
                            current_topic = extract_english_only(line_text_raw)
                        else:
                            current_section_name = extract_english_only(line_text_raw)
                        expecting_section_name = False
                        continue
                    expecting_section_name = False

                # 5. ARTICLE START DETECTION
                match_en = re_article_en.search(line_text_raw)
                match_ar = re_article_ar.search(line_text_raw)

                if match_en or match_ar:
                    if current_article:
                        extracted_articles.append(current_article)

                    art_num_str = match_en.group(1) if match_en else match_ar.group(1)
                    art_num = int(art_num_str.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")))

                    current_article = {
                        "article_number": art_num,
                        "book": current_book,
                        "chapter": current_chapter,
                        "section": current_section,
                        "section_name": current_section_name,
                        "topic": current_topic,
                        "text_ar": line_ar_str,
                        "text_en": line_en_str,
                        "is_repealed": "ملغاة" in line_text_raw or "repealed" in line_text_raw.lower(),
                        "source_page": page_num
                    }
                    continue

                # 6. TOPIC DETECTION
                is_numbered_topic = bool(re_topic_number.match(line_en_str) or re_topic_number.match(line_text_raw))

                if current_article is None or is_bold or is_numbered_topic:
                    if current_article:
                        extracted_articles.append(current_article)
                        current_article = None

                    topic_cand = extract_english_only(line_text_raw)
                    if topic_cand:
                        current_topic = topic_cand
                    continue

                # 7. CONTINUATION TEXT APPENDING TO ACTIVE ARTICLE
                if current_article is not None:
                    if line_ar_str:
                        current_article["text_ar"] = (current_article["text_ar"] + " " + line_ar_str).strip()
                    if line_en_str:
                        current_article["text_en"] = (current_article["text_en"] + " " + line_en_str).strip()

        if current_article:
            extracted_articles.append(current_article)

    # Deduplicate and sort articles
    unique_articles = {}
    for art in extracted_articles:
        num = art["article_number"]
        if num not in unique_articles:
            unique_articles[num] = art
        else:
            if art["text_ar"] and not unique_articles[num]["text_ar"]:
                unique_articles[num]["text_ar"] = art["text_ar"]
            if art["text_en"] and not unique_articles[num]["text_en"]:
                unique_articles[num]["text_en"] = art["text_en"]

    final_list = [unique_articles[k] for k in sorted(unique_articles.keys())]

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(final_list, f, ensure_ascii=False, indent=2)

    print(f"--- Completed! Extracted {len(final_list)} articles into '{output_json_path}' ---")
    return final_list

extract_legal_document(file, "output_4.json")

--- Starting Extraction: ../data/raw/legal_doc.pdf ---
Total Pages: 170
--- Completed! Extracted 1093 articles into 'output_4.json' ---


[{'article_number': 1,
  'book': '',
  'chapter': '',
  'section': 'SECTION I',
  'section_name': 'Laws and their Applications',
  'topic': '1. Laws and Rights -',
  'text_ar': 'مادة ١ ١ تسرى النصوص التشريعية على جميع المسائل التي تتناولها هذه النصوص في لفظها أو في فحواها. )٢( فإذا لم يوجد نص تشريعي يمكن تطبيقه، حكم القاضي بمقتضى العرف، فإذا لم يوجد، فبمقتضى مبادئ الشريعة اإلسالمية، فإذا لم يوجد فبمقتضى مبادئ القانون الطبيعي وقواعد العدالة',
  'text_en': 'Article 1 ( ( Provisions of laws govern all matters to which these ( ) provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. . In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.',
  'is_repealed': False,
  'source_page': 1},
 {'article_number': 2,
  'book': '',
  'chapter': '',
  'section': 'SECTION I',
  'secti